# LastSeen — Detector Comparison

Runs every candidate detector on the same 9 photos in `images/`, scores them on the same 16 query items
(8 inside COCO, 8 outside), and builds the comparison table + figures in `benchmark/results/`.

Models: YOLO11s (baseline), Faster R-CNN, RT-DETR-L, YOLO-World v2-S, YOLOE-11S (640 / 1280 px), YOLOE-11L, OWLv2.

**Run in Colab** (needs internet for weights from GitHub, PyTorch and Hugging Face). Running *all* models
here keeps the runtime column on the same hardware. CPU runtime is fine (~10–15 min total).

See `benchmark/COMPARISON.md` for the write-up and scoring details.

In [ ]:
# Setup: get the repo (skip if you already opened this notebook from inside the repo folder)
import os
if not os.path.exists("benchmark/compare_models.py"):
    !git clone https://github.com/IzzyHuang215/last_seen.git
    %cd last_seen
%pip install -q ultralytics transformers pandas matplotlib "git+https://github.com/ultralytics/CLIP.git"

## 1. Run all models
Raw detections are cached in `benchmark/results/raw/<model>.json`. Remove a model from the list to skip it.

If the script prints *"never hand-checked"* detections for a new model, look at those boxes and add them to
`benchmark/verified_boxes.json` (under `correct` or `wrong`) so they're scored properly.

In [ ]:
MODELS_TO_RUN = ["yolo11s", "fasterrcnn", "rtdetr", "yoloworld", "yoloe", "yoloe_1280", "yoloe_l", "owlv2"]
!python benchmark/compare_models.py --models {" ".join(MODELS_TO_RUN)}

## 2. Comparison table

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv("benchmark/results/summary.csv")
display(summary)
display(pd.read_csv("benchmark/results/per_item_ap.csv", index_col=0))

## 3. Figures

In [ ]:
!python benchmark/make_figures.py
display(Image(filename="benchmark/results/per_item_ap_heatmap.png"))
display(Image(filename="benchmark/results/example_9jpg.png"))

## 4. Per-model detections (boxes above each model's default threshold)
Green = COCO item, red = non-COCO item.

In [ ]:
MODEL = "owlv2"   # any key from MODELS_TO_RUN
for name in sorted(os.listdir(f"benchmark/results/vis/{MODEL}")):
    print(name)
    display(Image(filename=f"benchmark/results/vis/{MODEL}/{name}", width=700))